# Anatomy-Aware Pose Estimation on Lightweight Networks
**Studenti:** Alberto Messina, Flavio Massaroni — AI & Robotics Engineering, Sapienza
**Dataset:** COCO 2017 Keypoints (train) · OCHuman (zero-shot eval)
**Modello finale:** sev\_E7 — MobileNetV3-Small + STL con severity weighting (SEVERITY\_ALPHA=2.0)

**Setup Kaggle (prima di eseguire):**
1. Settings → **Internet: On**
2. Add Input → COCO 2017 Keypoints (asad11914) · OCHuman (messinaalberto) · pose-baseline-checkpoint (messinaalberto)


## Imports

In [ ]:
import os
import sys
from pathlib import Path

os.environ["POSE_ENV"] = "local"

# Usa il repository locale se presente nel workspace
repo_root = Path.cwd()
candidates = [
    repo_root,
    repo_root / "code",
    Path("/home/flavio/Documents/University/Computer vision/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks"),
    Path("/home/flavio/Documents/University/Computer vision/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks/code"),
]

mod_dir = None
for candidate in candidates:
    if (candidate / "config.py").exists():
        mod_dir = str(candidate.resolve())
        break

if not mod_dir:
    # Fallback per Kaggle
    REPO_URL = "https://github.com/flaviomassaroni/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks.git"
    REPO_DIR = "/kaggle/working/repo"

    !rm -rf {REPO_DIR}
    !git clone -q {REPO_URL} {REPO_DIR}

    for root, dirs, files in os.walk(REPO_DIR):
        if ".git" in root:
            continue
        if "config.py" in files:
            mod_dir = root
            break

if mod_dir:
    sys.path.insert(0, mod_dir)
    print("Modulo root:", mod_dir)
    print("Moduli:", sorted(f for f in os.listdir(mod_dir) if f.endswith(".py")))
else:
    raise RuntimeError("config.py non trovato. Controlla il percorso locale o l'accesso a Internet.")

# Import moduli progetto
from config import *
import utils, data, network, train
import evaluation as ev
from stl import SkeletalTopologyLoss, calibrate_lambdas
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import cv2

cv2.setNumThreads(0)  # evita conflitti con DataLoader multiprocess
set_seed(SEED)
print(f"Device: {device} | Ambiente: {ENV_NAME} | SEED: {SEED}")


## Globals

In [ ]:
import config, importlib
importlib.reload(config)
from config import *

BEST_PTH = config.BEST_PTH

print("=== Configurazione ===")
print(f"  BEST_PTH       : {BEST_PTH}")
print(f"  BEST_PTH esiste: {os.path.exists(BEST_PTH)}")
print(f"  CHECKPOINT_DIR : {CHECKPOINT_DIR}")
print(f"  RESULTS_DIR    : {RESULTS_DIR}")
print(f"  COCO val       : {COCO_VAL_ANN}")
print(f"  OCHuman val    : {OCHUMAN_VAL_ANN}")
print(f"")
print(f"=== Iperparametri STL (sev_E7) — tutti da config.py ===")
print(f"  STL_BETA         = {STL_BETA}")
print(f"  SEVERITY_ALPHA   = {SEVERITY_ALPHA}")
print(f"  STL_FINE_TUNE_LR = {STL_FINE_TUNE_LR}")
print(f"  STL_BONE_BOOST   = {STL_BONE_BOOST}")
print(f"  STL_SEV_EPOCHS   = {STL_SEV_EPOCHS}")
print(f"  STL_TARGET_FRAC  = {STL_TARGET_FRAC}")

if os.path.isdir("/kaggle/input"):
    print("\n/kaggle/input:")
    for d in os.listdir("/kaggle/input"):
        print(f"  - {d}")


## Utils

In [ ]:
# Le funzioni di supporto sono in utils.py
import inspect

print("Funzioni disponibili in utils.py:")
for name in ["generate_heatmap", "decode_heatmaps", "heatmap_to_original", "count_params"]:
    fn = getattr(utils, name)
    sig = inspect.signature(fn)
    print(f"  utils.{name}{sig}")


## Data

In [ ]:
# Dataset COCO Keypoints: train (116k campioni, filtro >= 8 kp) + val (6k)
# Preprocessing: aspect-ratio preserving resize + padding centrato.
# Due loader: standard (num_workers=2, per baseline training if False:) e deterministico

train_samples = data.build_samples(COCO_TRAIN_ANN, min_keypoints=8)
val_samples   = data.build_samples(COCO_VAL_ANN)
print(f"COCO: {len(train_samples):,} train | {len(val_samples):,} val")

train_dataset = data.COCOKeypointsDataset(
    train_samples, COCO_TRAIN_IMG, INPUT_SIZE, HEATMAP_SIZE, SIGMA, NUM_KEYPOINTS)
val_dataset = data.COCOKeypointsDataset(
    val_samples, COCO_VAL_IMG, INPUT_SIZE, HEATMAP_SIZE, SIGMA, NUM_KEYPOINTS)

# Loader standard — usato solo nel baseline training 
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          num_workers=2, pin_memory=True)

# Loader deterministico — per calibrazione + training severity
g = torch.Generator()
g.manual_seed(SEED)
train_loader_det = DataLoader(train_dataset, batch_size=32, shuffle=True,
                               num_workers=0, pin_memory=True, generator=g)

# Campioni OCHuman per valutazione zero-shot
oc_samples = data.build_samples(OCHUMAN_VAL_ANN)
print(f"OCHuman: {len(oc_samples):,} campioni (zero-shot eval)")
print(f"Loader: {len(train_loader)} batch/std | {len(train_loader_det)} batch/det")

## Network

In [ ]:
# Architettura: MobileNetV3-Small (backbone) + DeconvHead (3x ConvTranspose2d, 256 ch)

# BASELINE TRAINING
if False:
    val_loader_base = DataLoader(val_dataset, batch_size=32, shuffle=False,
                                 num_workers=2, pin_memory=True)
    model_b = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=True).to(device)
    opt_b   = torch.optim.Adam(model_b.parameters(), lr=1e-3)
    sch_b   = torch.optim.lr_scheduler.MultiStepLR(opt_b, milestones=[15, 25], gamma=0.1)
    train.fit(model_b, train_loader, val_loader_base, opt_b, sch_b,
              train.WeightedMSELoss(), device, 30, CHECKPOINT_DIR, resume=True)

# Carica checkpoint baseline
_best = BEST_PTH
if not _best or not os.path.exists(_best):
    _best = "/kaggle/input/datasets/messinaalberto/pose-baseline-checkpoint/best.pth"
assert os.path.exists(_best), f"best.pth non trovato: {_best}"

model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model.load_state_dict(torch.load(_best, map_location=device))

total_p, train_p = utils.count_params(model)
print(f"Baseline caricato: {_best}")
print(f"Parametri: {total_p:,} totali | {train_p:,} trainabili")


## Train

In [ ]:
# Fine-tuning STL con severity weighting — modello sev_E7
# Tutti gli iperparametri vengono da config.py
# SEVERITY_ALPHA=2.0 e' gia' attivo in bone_ratio_loss (importato da config in stl.py)
# pesa la hinge simmetria per (1 + alpha*excess), rinforza i violatori correggibili
# senza accanirsi sul foreshortening floor.

import time

# Ricarichiamo il baseline da zero
model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model.load_state_dict(torch.load(_best, map_location=device))

# Calibrazione lambda gradient-norm (GradNorm-style, statica) 
# Misura su train_loader_det (deterministico).
criterion = SkeletalTopologyLoss(
    heatmap_criterion=train.WeightedMSELoss(),
    lambda_bone=1.0, lambda_angle=1.0, lambda_order=1.0, lambda_collapse=1.0,
    beta=STL_BETA,
)
print("Calibrazione lambda (gradient-norm, n_batches=4)...")
calibrate_lambdas(criterion, model, train_loader_det, device,
                  target_frac=STL_TARGET_FRAC, n_batches=4, verbose=True)

# Boost bone post-calibrazione (compensa diluizione gradiente)
lam_bone_calib = criterion.lambda_bone
criterion.lambda_bone = lam_bone_calib * STL_BONE_BOOST
print(f"lambda_bone: {lam_bone_calib:.5f} -> {criterion.lambda_bone:.5f} ({STL_BONE_BOOST:.0f}x boost)")

# Azzera gradienti sporchi lasciati dalla calibrazione
optimizer = torch.optim.Adam(model.parameters(), lr=STL_FINE_TUNE_LR)
optimizer.zero_grad(set_to_none=True)

SEV_DIR = os.path.abspath(os.path.join(CHECKPOINT_DIR, "..", f"sev_lin_a{SEVERITY_ALPHA}"))
os.makedirs(SEV_DIR, exist_ok=True)
print(f"\nOutput: {SEV_DIR}")
print(f"{'ep':>3} {'AP':>7} {'AVR':>7} {'bone':>7} {'angle':>7} {'colla':>7}  "
      f"{'L_bone':>8} {'L_hm':>8}  time  note")
print("-" * 80)

# E0: baseline prima di qualsiasi fine-tuning (reference )
model.eval()
_, avr0 = ev.evaluate_on_coco_val(model, val_samples, device,
                                   results_path=os.path.join(SEV_DIR, "pred_e00.json"))
pc0 = avr0["per_category"]
print(f"{'0':>3} {'-':>7} {avr0['AVR_pose_rate']:>7.4f} "
      f"{pc0['bone_ratio']:>7.4f} {pc0['joint_angle']:>7.4f} "
      f"{pc0['collapse']:>7.4f}   (baseline E0)")

best = (avr0["AVR_pose_rate"], 0, None)
history = []

for epoch in range(1, STL_SEV_EPOCHS + 1):
    t0 = time.time()
    model.train()
    sums = {k: 0.0 for k in ["heatmap", "bone", "total"]}

    for imgs, hms, w in tqdm(train_loader_det, desc=f"sev E{epoch:02d}", leave=False):
        imgs, hms, w = imgs.to(device), hms.to(device), w.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss, terms = criterion(out, hms, w)
        loss.backward()
        optimizer.step()
        for k in sums:
            sums[k] += terms[k] * imgs.size(0)
    n = len(train_loader_det.dataset)
    for k in sums:
        sums[k] /= n

    torch.save(model.state_dict(), os.path.join(SEV_DIR, f"epoch_{epoch:02d}.pth"))

    model.eval()
    ce, avr = ev.evaluate_on_coco_val(
        model, val_samples, device,
        results_path=os.path.join(SEV_DIR, f"pred_e{epoch:02d}.json"))
    ap = float(ce.stats[0])
    pc = avr["per_category"]
    avr_rate = avr["AVR_pose_rate"]

    note = ""
    if avr_rate < best[0] and ap >= 0.486:
        best = (avr_rate, epoch, ap)
        note = " *best"

    print(f"{epoch:>3} {ap:>7.4f} {avr_rate:>7.4f} "
          f"{pc['bone_ratio']:>7.4f} {pc['joint_angle']:>7.4f} "
          f"{pc['collapse']:>7.4f}  {sums['bone']:>8.4f} {sums['heatmap']:>8.4f}  "
          f"{time.time()-t0:.0f}s{note}")
    history.append({"epoch": epoch, "AP": ap, "AVR": avr_rate, **pc})

print(f"\nmiglior AVR COCO: {best[0]:.4f} @E{best[1]} (AP {best[2]:.4f})")
SELECTED_EPOCH = best[1]  # passato alla cella Evaluation
print(f"Checkpoint: {os.path.join(SEV_DIR, f'epoch_{SELECTED_EPOCH:02d}.pth')}")


## Evaluation

In [ ]:
# Valutazione finale deterministica: baseline vs sev_E7 su COCO val + OCHuman
# evaluation.run_inference (deterministico)
# OCHuman: valutazione zero-shot
import json

SEV_DIR  = os.path.abspath(os.path.join(CHECKPOINT_DIR, "..", f"sev_lin_a{SEVERITY_ALPHA}"))
EVAL_DIR = os.path.abspath(os.path.join(CHECKPOINT_DIR, "..", "eval_sev_E7"))
os.makedirs(EVAL_DIR, exist_ok=True)

try:
    _epoch = SELECTED_EPOCH  
except NameError:
    _epoch = 7               # fallback: sev_E7 e' il checkpoint documentato

MODELS_TO_EVAL = {
    "baseline": _best,
    f"sev_E{_epoch}": os.path.join(SEV_DIR, f"epoch_{_epoch:02d}.pth"),
}
print("Checkpoint da valutare:")
for name, path in MODELS_TO_EVAL.items():
    print(f"  {name:12} {'OK' if os.path.exists(path) else 'MANCANTE'}  {path}")

def load_model(path):
    m = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
    m.load_state_dict(torch.load(path, map_location=device))
    m.eval()
    return m

rows = {}
for name, path in MODELS_TO_EVAL.items():
    if not os.path.exists(path):
        print(f"\n!! {name}: checkpoint mancante, salto")
        continue
    print(f"\n===== {name} =====")
    m = load_model(path)

    print("-- COCO val (num_workers=0, deterministico) --")
    ce_c, avr_c = ev.evaluate_on_coco_val(
        m, val_samples, device,
        results_path=os.path.join(EVAL_DIR, f"{name}_coco.json"))

    print("-- OCHuman zero-shot (num_workers=0, deterministico) --")
    ce_o, avr_o = ev.evaluate_on_ochuman(
        m, device, results_path=os.path.join(EVAL_DIR, f"{name}_oc.json"))

    rows[name] = {
        "coco_ap":  float(ce_c.stats[0]),
        "coco_avr": avr_c["AVR_pose_rate"],
        "coco_pc":  avr_c["per_category"],
        "oc_ap":    float(ce_o.stats[0]),
        "oc_avr":   avr_o["AVR_pose_rate"],
        "oc_pc":    avr_o["per_category"],
        "oc_mean":  avr_o["AVR_mean_violations"],
    }
    del m
    torch.cuda.empty_cache()

# Tabella riassuntiva
print("\n" + "=" * 70)
print("RISULTATI FINALI — Anatomy-Aware Pose Estimation")
print("=" * 70)
print(f"{'modello':>12} | {'COCO AP':>8} {'COCO AVR':>9} | {'OC AP':>7} {'OC AVR':>8} {'OC mean':>8}")
print("-" * 70)
base = rows.get("baseline")
for name, r in rows.items():
    print(f"{name:>12} | {r['coco_ap']:>8.4f} {r['coco_avr']:>9.4f} | "
          f"{r['oc_ap']:>7.4f} {r['oc_avr']:>8.4f} {r['oc_mean']:>8.4f}")
print("-" * 70)

sev_name = f"sev_E{_epoch}"
if base and sev_name in rows:
    s = rows[sev_name]
    coco_avr_d = s["coco_avr"] - base["coco_avr"]
    oc_avr_d   = s["oc_avr"]   - base["oc_avr"]
    print(f"\nDELTA {sev_name} vs baseline (negativo AVR = miglioramento):")
    print(f"  COCO AP  : {s['coco_ap']  - base['coco_ap']:+.4f}")
    print(f"  COCO AVR : {coco_avr_d:+.4f}  {'(MIGLIORE)' if coco_avr_d < 0 else '(peggiore)'}")
    print(f"  OC   AP  : {s['oc_ap']    - base['oc_ap']:+.4f}")
    print(f"  OC   AVR : {oc_avr_d:+.4f}  {'(MIGLIORE)' if oc_avr_d < 0 else '(peggiore)'}")

print("\nAVR per-categoria su OCHuman (rate):")
print(f"{'modello':>12} | {'bone_ratio':>10} {'joint_angle':>12} {'collapse':>9}")
print("-" * 50)
for name, r in rows.items():
    pc = r["oc_pc"]
    print(f"{name:>12} | {pc['bone_ratio']:>10.4f} {pc['joint_angle']:>12.4f} "
          f"{pc['collapse']:>9.4f}")

out_json = {k: {kk: vv for kk, vv in v.items() if kk not in ("oc_pc", "coco_pc")}
            for k, v in rows.items()}
with open(os.path.join(EVAL_DIR, "results_sev_E7.json"), "w") as f:
    json.dump(out_json, f, indent=2)
print(f"\nSalvato: {EVAL_DIR}/results_sev_E7.json")
print("=" * 70)
